In [77]:
import geopandas as gpd
import pandas as pd

In [78]:
setores_allevant = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais enviados pelo cliente (DOX)\Allevant\04 - Revisão 01\Setores de projeto_Bairros_Rural.shp').to_crs('EPSG:4326')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais enviados pelo cliente (DOX)\Base GIS\Base GIS\Vetoriais\area_abrangencia_V2.gpkg').to_crs('EPSG:4326')
dom = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais baixados online\4204301\4204301.csv', delimiter = ';')
setores_ibge = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\SC_setores_CD2022.gpkg').to_crs('EPSG:4326')

dom_particular = dom[dom['COD_ESPECIE']==1]
dom_particular_gdf = gpd.GeoDataFrame(
    dom_particular,
    geometry=gpd.points_from_xy(dom_particular['LONGITUDE'], dom_particular['LATITUDE']),
    crs="EPSG:4326"
)

In [79]:
#Área de análise - Área que mescla allevant + aps
area_analise = gpd.overlay(aps, setores_allevant, how="union").dissolve()

dom_analise = gpd.clip(dom_particular_gdf,area_analise)


C:\Users\gabriel.coimbra\AppData\Local\Temp\ipykernel_20284\1289016222.py:2: UserWarning: `keep_geom_type=True` in overlay resulted in 521 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  area_analise = gpd.overlay(aps, setores_allevant, how="union").dissolve()


In [80]:
#Domicilios em cada setor da allevant
setores_allevant['AGRUP'] = setores_allevant['AGRUP'].fillna('Rural (setor vazio)')

dom_setores = gpd.sjoin(
    dom_analise,
    setores_allevant[['nome','AGRUP', 'geometry']],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

dom_setores['AGRUP'] = dom_setores['AGRUP'].fillna('Fora da setorização')

contagem = (
    dom_setores
    .groupby('AGRUP')
    .size()
    .reset_index(name='qtd_domicilios')
)


In [81]:
aps['dentro'] = 'Dentro da APS'
dom_setores_aps = gpd.sjoin(
    dom_setores,
    aps[['geometry','dentro']],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

dom_setores_aps['dentro'] = dom_setores_aps['dentro'].fillna('Fora da APS')

dom_setores_aps

,COD_UF,COD_MUN,COD_ESPECIE,LATITUDE,LONGITUDE,NV_GEO_COORD,geometry,nome,AGRUP,dentro
21598,42,4204301,1,-27.261868,-52.016921,1,POINT (-52.01692 -27.26187),None,ZEIS,Dentro da APS
21516,42,4204301,1,-27.261837,-52.017012,1,POINT (-52.01701 -27.26184),None,ZEIS,Dentro da APS
21514,42,4204301,1,-27.261783,-52.017242,1,POINT (-52.01724 -27.26178),None,ZEIS,Dentro da APS
21600,42,4204301,1,-27.261671,-52.017222,1,POINT (-52.01722 -27.26167),None,ZEIS,Dentro da APS
21599,42,4204301,1,-27.261602,-52.017215,1,POINT (-52.01722 -27.2616),None,ZEIS,Dentro da APS
...,...,...,...,...,...,...,...,...,...,...
38063,42,4204301,1,-27.180599,-51.916049,1,POINT (-51.91605 -27.1806),None,ZI3,Dentro da APS
38066,42,4204301,1,-27.180537,-51.918867,1,POINT (-51.91887 -27.18054),None,ZI3,Dentro da APS
38061,42,4204301,1,-27.180431,-51.915448,1,POINT (-51.91545 -27.18043),None,ZI3,Dentro da APS
38065,42,4204301,1,-27.180414,-51.917764,1,POINT (-51.91776 -27.18041),None,ZI3,Dentro da APS


In [82]:
aps['dentro'] = 'Dentro da APS'
dom_setores_aps_ibge = gpd.sjoin(
    dom_setores_aps,
    setores_ibge[['SITUACAO','geometry']],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

dom_setores_aps_ibge['label'] = dom_setores_aps_ibge['AGRUP'] + ' ' +dom_setores_aps_ibge['dentro'] + ' - IBGE: ' + dom_setores_aps_ibge['SITUACAO']

In [83]:
dom_setores_aps_ibge['label'].value_counts()
dom_setores_aps_ibge.to_file(r"C:\Users\gabriel.coimbra\Desktop\Concórdia\dom_setores_aps_ibge_2.gpkg", driver="GPKG")
